### Basic data exploration, cleaning, and resampling of iSUPER 2023, 2024, and 2025 (Jan-Aug) dataset

### EDA on dataset

In [ ]:
# import libraries
import pandas as pd
import numpy as np

# list of input files (add more file if required)
files = [
    "iSUPER data/Chelsea DEP MOD-00247 Full 2023.csv",
    "iSUPER data/Chelsea DEP MOD-00247 Full 2024.csv",
    "iSUPER data/Chelsea DEP MOD-00214 Jan-Aug 2025.csv",
]

# read and stack all the years together
df_list = []

for file in files:
    df_raw = pd.read_csv(file)
    print(f"Loaded {file} with shape {df_raw.shape}")
    # Rename columns -> because we have different flag name spacing in different files
    rename_dict = {}
    for col in df_raw.columns:
        if col.replace(" ", "").lower() == "pm10>150?":
            rename_dict[col] = "pm10 > 150?"
    if rename_dict:
        df_raw = df_raw.rename(columns=rename_dict)

    df_list.append(df_raw)

# concatenate
df_raw = pd.concat(df_list, ignore_index=True)

display(df_raw[2000:3000])
print(df_raw.dtypes)
df_raw.shape

Loaded iSUPER data/Chelsea DEP MOD-00247 Full 2023.csv with shape (8760, 16)
Loaded iSUPER data/Chelsea DEP MOD-00247 Full 2024.csv with shape (8784, 16)
Loaded iSUPER data/Chelsea DEP MOD-00214 Jan-Aug 2025.csv with shape (5471, 16)


,period_start,period_end,period_start_utc,period_end_utc,sn,n_datapoints,rh,temp,pm1,pm25,pm10,co,no,no2,o3,pm10 > 150?
2000,2023-03-25T09:00:00-04:00,2023-03-25T10:00:00-04:00,2023-03-25T13:00:00+00:00,2023-03-25T14:00:00+00:00,MOD-00247,60,55.838,6.542,2.073,3.691,33.041,NaN,NaN,NaN,NaN,0.0
2001,2023-03-25T10:00:00-04:00,2023-03-25T11:00:00-04:00,2023-03-25T14:00:00+00:00,2023-03-25T15:00:00+00:00,MOD-00247,60,51.167,7.000,1.738,3.106,27.479,NaN,NaN,NaN,NaN,0.0
2002,2023-03-25T11:00:00-04:00,2023-03-25T12:00:00-04:00,2023-03-25T15:00:00+00:00,2023-03-25T16:00:00+00:00,MOD-00247,60,53.183,6.655,1.789,3.455,33.886,NaN,NaN,NaN,NaN,0.0
2003,2023-03-25T12:00:00-04:00,2023-03-25T13:00:00-04:00,2023-03-25T16:00:00+00:00,2023-03-25T17:00:00+00:00,MOD-00247,60,54.830,6.720,1.703,3.329,30.645,NaN,NaN,NaN,NaN,0.0
2004,2023-03-25T13:00:00-04:00,2023-03-25T14:00:00-04:00,2023-03-25T17:00:00+00:00,2023-03-25T18:00:00+00:00,MOD-00247,60,56.635,6.840,1.553,3.398,37.372,NaN,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,2023-05-05T20:00:00-04:00,2023-05-05T21:00:00-04:00,2023-05-06T00:00:00+00:00,2023-05-06T01:00:00+00:00,MOD-00247,60,66.182,10.838,2.668,4.492,66.262,343.531,2.634,46.722,19.056,0.0
2996,2023-05-05T21:00:00-04:00,2023-05-05T22:00:00-04:00,2023-05-06T01:00:00+00:00,2023-05-06T02:00:00+00:00,MOD-00247,60,73.560,9.977,4.018,5.719,56.261,467.239,1.850,44.646,16.811,0.0
2997,2023-05-05T22:00:00-04:00,2023-05-05T23:00:00-04:00,2023-05-06T02:00:00+00:00,2023-05-06T03:00:00+00:00,MOD-00247,60,76.587,9.813,4.602,5.989,47.157,563.524,2.498,42.510,15.649,0.0
2998,2023-05-05T23:00:00-04:00,2023-05-06T00:00:00-04:00,2023-05-06T03:00:00+00:00,2023-05-06T04:00:00+00:00,MOD-00247,60,70.018,11.012,4.285,4.803,12.731,314.003,2.005,34.902,20.993,0.0


period_start         object
period_end           object
period_start_utc     object
period_end_utc       object
sn                   object
n_datapoints          int64
rh                  float64
temp                float64
pm1                 float64
pm25                float64
pm10                float64
co                  float64
no                  float64
no2                 float64
o3                  float64
pm10 > 150?         float64
dtype: object


(23015, 16)

### Re-sampling and cleaning

We will focus on the some columns in this dataset, hourly sampling is also done (not required for this dataset though but its good to be safe). Missing data is present here, this should be dealt after merging with other datasets.

In [160]:
# rename period_start_utc to timestamp_utc
df_raw["timestamp_utc"] = pd.to_datetime(df_raw["period_start_utc"], utc=True)

In [161]:
# keep only relevant columns
df_cleaned = df_raw[[
    "timestamp_utc",
    "temp",
    "rh",
    "pm1",
    "pm25",
    "pm10",
    "pm10 > 150?",
]].copy()

display(df_cleaned.head(10))
df_cleaned.shape

,timestamp_utc,temp,rh,pm1,pm25,pm10,pm10 > 150?
0,2023-01-01 05:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-01 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-01 07:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-01 08:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-01 09:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
5,2023-01-01 10:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
6,2023-01-01 11:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
7,2023-01-01 12:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
8,2023-01-01 13:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
9,2023-01-01 14:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN


(23015, 7)

In [162]:
df_cleaned = df_cleaned.set_index("timestamp_utc").sort_index()
df_cleaned.head(5)

,temp,rh,pm1,pm25,pm10,pm10 > 150?
timestamp_utc,,,,,,
2023-01-01 05:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-01 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-01 07:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-01 08:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-01 09:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN


In [163]:
# In this dataset, out data are already hourly, but we can still resample to enforce a clean hourly format. 
# Keep it consistent with other datasets cleaning script.
agg_rules = {
    "rh": "mean",
    "temp": "mean",
    "pm1": "mean",
    "pm25": "mean",
    "pm10": "mean",
    "pm10 > 150?": "max", # Here, we use max so that "any exceedance within an hour" would be flag as "1"
}

isuper_hourly = df_cleaned.resample("1H").agg(agg_rules)

C:\Users\USER\AppData\Local\Temp\ipykernel_32596\625059651.py:12: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  isuper_hourly = df_cleaned.resample("1H").agg(agg_rules)


In [166]:
isuper_hourly.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 23015 entries, 2023-01-01 05:00:00+00:00 to 2025-08-17 03:00:00+00:00
Freq: h
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   rh           20257 non-null  float64
 1   temp         20257 non-null  float64
 2   pm1          20190 non-null  float64
 3   pm25         20190 non-null  float64
 4   pm10         20190 non-null  float64
 5   pm10 > 150?  20190 non-null  float64
dtypes: float64(6)
memory usage: 1.2 MB


In [ ]:
isuper_hourly[isuper_hourly["pm10 > 150?"] == 1].sample(10)


,rh,temp,pm1,pm25,pm10,pm10 > 150?
timestamp_utc,,,,,,
2023-04-23 12:00:00+00:00,81.763,9.867,1.664,15.057,150.213,1.0
2023-10-07 11:00:00+00:00,86.045,19.600,2.784,16.344,179.028,1.0
2024-01-13 08:00:00+00:00,82.905,6.832,3.146,15.420,156.237,1.0
2023-12-26 03:00:00+00:00,91.965,4.427,7.389,18.793,3263.258,1.0
2024-01-23 12:00:00+00:00,45.632,2.953,7.859,11.792,164.707,1.0
2023-07-10 11:00:00+00:00,89.712,21.528,1.202,2.278,845.422,1.0
2024-03-21 20:00:00+00:00,21.590,8.092,2.951,11.542,699.593,1.0
2024-06-07 00:00:00+00:00,83.760,18.672,3.091,3.514,189.571,1.0
2024-02-05 12:00:00+00:00,49.672,-2.470,3.779,14.288,240.630,1.0


In [165]:
# save csv file as isuper_hourly
isuper_hourly.to_csv("iSUPER data/isuper_hourly_full.csv")